# Misinformation Detection and Fact-Checking System

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://img.etimg.com/thumb/msid-72106572,width-480,height-360,imgsize-216496,resizemode-75/seven-types-of-fake-news.jpg"> 
</p>
</div>

## Description:

This code defines a system for detecting misinformation in news articles using a language model (LLM). It fetches the article, analyzes its authenticity, and returns a detailed response, including:


- Misinformation score: The likelihood of the news being fake.

- Explanation: Reasoning behind the score.

- Linguistic features: Key language indicators suggesting misinformation.

- Fact-checking results: Verifying the factual accuracy of claims in the article.

The system utilizes classes like `MisinformationDetector` and `FactCheckResult` to process and return actionable insights on news articles, helping to identify fake or misleading content.





## Step 1: Environment Setup and Installation

This cell handles initial setup for the notebook:

- Installs dependencies from `requirements/fake_news_and_misinformation_detector.requirements.txt`.

- Retries installation up to 3 times on failure.

- Loads environment variables from `.env` using `python-dotenv`.

- Ensures `OPENAI_API_KEY` is set before continuing.

After setup, it clears the output and confirms success.


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
from dotenv import load_dotenv
import os

requirements_installed = False
max_retries = 3
retries = 0
REQUIRED_ENV_VARS = ["OPENAI_API_KEY"]
PROJECT_NAME = "fake_news_and_misinformation_detector"
REQUIREMENTS_FILE = f"{PROJECT_NAME}.requirements.txt"


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system(f"pip install -r requirements/{REQUIREMENTS_FILE}")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)


install_requirements()
clear_output()
setup_env()
print("🚀 Setup complete. Continue to the next cell.")

## Step 2: Environment Variable Setup


- The first line imports the `FirecrawlApp` class from the `firecrawl` library, which is likely used for web scraping or fetching data from URLs.

- The second line defines a function `get_firecrawl_client`, which is responsible for creating and returning a client instance of the `FirecrawlApp`.

- The third line fetches the API key from the environment variable `FIRECRAWL_API_KEY` using `os.getenv`, and then initializes the `FirecrawlApp` with this API key to allow interaction with the `firecrawl` API.


In [ ]:
from firecrawl import FirecrawlApp


def get_firecrawl_client():
    return FirecrawlApp(api_key=os.getenv("FIRECRAWL_API_KEY"))

## Step 3: Fetching and Validating News Articles


The `NewsFetcher` class scrapes and verifies if a URL contains a valid news article.


- Fetches news articles from a URL in markdown format.

- Uses a language model to determine if the content is a news article.

- Utilizes a web crawler client to scrape the content.

- Handles errors with exception handling and detailed traceback.

- Returns the markdown content of a valid news article, or `None` if invalid.



In [ ]:
import traceback
from typing import Union
from litellm import completion


class NewsFetcher:
    """
    Fetches news articles from the given URL.

    """

    DEFAULT_LLM_MODEL = "anthropic/claude-3-5-sonnet-latest"
    AUSPICIOUS_NUMBER = 11
    DEFAULT_TEMPERATURE = 1 - AUSPICIOUS_NUMBER

    def __init__(self, model=DEFAULT_LLM_MODEL):
        self.crawler = get_firecrawl_client()
        self.model = model

    def is_news_article(self, page_markdown: str) -> bool:
        """
        Checks if the given URL is a news article.

        Args:
            url (str): The URL to check.

        Returns:
            bool: True if the URL is a news article, False otherwise.
        """
        try:

            user_prompt = f"""
                Is this a news article?
                Page: '''{page_markdown}'''.
                Respond with 0 for No and 1 for Yes and nothing else.
            """
            response = completion(
                model=self.model,
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt,
                    }
                ],
            )
            result = response.choices[0].message.content

            if "0" in result:
                return False

            if "1" in result:
                return True

            raise Exception("Failed to determine if the URL is a news article.")
        except Exception as e:
            print(f"Failed to check if the URL is a news article: {e}")
            traceback.print_exc()
            return False

    def get_news_article(self, url: str) -> Union[str, None]:
        """
        Fetches the news article from the given URL in markdown format.

        Args:
            url (str): The URL of the news article.

        Returns:
            str: The news article in markdown format.
        """
        try:
            app = get_firecrawl_client()
            scrape_result = app.scrape_url(url, params={"formats": ["markdown"]})
            if not scrape_result:
                return None

            success = scrape_result["metadata"]["statusCode"] == 200

            if not success:
                print(f"Failed to get docs from URL: {url}")
                return None

            page_markdown = scrape_result["markdown"]

            if not self.is_news_article(page_markdown):
                print("The given URL is not a news article.")
                return None

            return page_markdown
        except Exception as e:
            print(f"Failed to fetch news article: {e}")
            traceback.print_exc()
            return None

## Step 4: Displaying the Fetched News Article

- Imports `Markdown` and `display` from `IPython.display` to display formatted markdown in a Jupyter notebook.

- Creates an instance of the `NewsFetcher` class.

- Sets the URL of a news article from the Indian Express.

- Calls `get_news_article` method to fetch the article's markdown content from the given URL.

- If the article is successfully fetched, it is displayed as markdown. If not, an error message is printed indicating the failure.



In [ ]:
from IPython.display import Markdown, display

news_fetcher = NewsFetcher()

url = "https://indianexpress.com/article/education/cbse-dual-board-exam-class-10-2026-9856208/"

markdown = news_fetcher.get_news_article(url)

if markdown:
    display(Markdown(markdown))
else:
    print("Failed to fetch the news article.")

## Step 5: Fake News Detection

- **FakeNewsDetectionResponse**: A Pydantic model that structures the response with a boolean `is_fake` and an `explanation`.

- **FakeNewsDetector**: A class to detect fake news using an LLM model.

  - It fetches the article using the `NewsFetcher` class.

  - Sends a prompt to the LLM model asking if the article is fake and retrieves a response.

  - The response is parsed and returned as a structured `FakeNewsDetectionResponse`.

- **Error Handling**: Logs and raises an error if the article fetching or model interaction fails.


In [ ]:
import traceback
from litellm import completion, supports_response_schema
from pydantic import BaseModel
import json


class FakeNewsDetectionResponse(BaseModel):
    """
    Response from the fake news detection API.
    """

    is_fake: bool
    explanation: str


class FakeNewsDetector:
    """
    Fake news detector using the LLM model.
    """

    DEFAULT_LLM_MODEL = "anthropic/claude-3-5-sonnet-latest"
    AUSPICIOUS_NUMBER = 0.108
    DEFAULT_TEMPERATURE = 1 - AUSPICIOUS_NUMBER

    def __init__(self, model=DEFAULT_LLM_MODEL, news_fetcher_model=DEFAULT_LLM_MODEL):
        self.model = model
        self.news_fetcher = NewsFetcher(model=news_fetcher_model)

    def detect_fake_news(self, url: str) -> FakeNewsDetectionResponse:
        """
        Determines if the given news article is fake.

        Args:
            news_article (str): The news article in markdown format.

        Returns:
            bool: True if the news article is fake, False otherwise.
        """
        try:
            # assert supports_response_schema(model=self.DEFAULT_LLM_MODEL)
            news_article = self.news_fetcher.get_news_article(url)
            if not news_article:
                print("Failed to fetch the news article.")
                raise Exception("Failed to fetch the news article.")

            user_prompt = f"""
                Is this news article fake?
                Article: '''{news_article}'''.
                Respond with 0 for No and 1 for Yes and nothing else.
                Also provide a brief explanation.
            """
            response = completion(
                model=self.model,
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt,
                    }
                ],
                response_format=FakeNewsDetectionResponse,
            )
            result_json = response.choices[0].message.content
            result = json.loads(result_json)
            return FakeNewsDetectionResponse(**result)
        except Exception as e:
            print(f"Failed to check if the news article is fake: {e}")
            traceback.print_exc()
            raise e

## Step 6: Misinformation Detection

- **FactCheckResult**: A Pydantic model representing the result of a fact check with `fact`, `authenticity_score`, and `explanation`.

- **MisinformationDetectionResponse**: A Pydantic model that represents the response from the misinformation detection API, including:

  - `misinformation_score`: A score indicating the likelihood of misinformation.

  - `explanation`: Explanation of the misinformation score.

  - `linguistic_features`: Features that indicate misinformation.

  - `fact_check`: A list of fact check results.



- **MisinformationDetector**: A class for detecting misinformation in news articles using an LLM model.

  - It fetches the article using `NewsFetcher`.

  - Sends a detailed prompt to the LLM model to detect misinformation.

  - The response includes the misinformation score, explanation, linguistic features, and fact checks, which are returned as a `MisinformationDetectionResponse`.



- **Error Handling**: Handles errors in fetching the article or processing the model response, logs the error, and raises an exception.



In [ ]:
import traceback
from litellm import completion, supports_response_schema
from pydantic import BaseModel
import json


class FactCheckResult(BaseModel):
    """
    Fact check result.
    """

    fact: str
    authenticity_score: float
    explanation: str


class MisinformationDetectionResponse(BaseModel):
    """
    Response from the fake news detection API.
    """

    misinformation_score: float
    explanation: str
    linguistic_features: str
    fact_check: list[FactCheckResult]


class MisinformationDetector:
    """
    Fake news detector using the LLM model.
    """

    DEFAULT_LLM_MODEL = "anthropic/claude-3-5-sonnet-latest"
    AUSPICIOUS_NUMBER = 0.108
    DEFAULT_TEMPERATURE = 1 - AUSPICIOUS_NUMBER

    def __init__(self, model=DEFAULT_LLM_MODEL, news_fetcher_model=DEFAULT_LLM_MODEL):
        self.model = model
        self.news_fetcher = NewsFetcher(model=news_fetcher_model)

    def detect_misinformation(self, url: str) -> FakeNewsDetectionResponse:
        """
        Determines if the given news article is fake.

        Args:
            news_article (str): The news article in markdown format.

        Returns:
            bool: True if the news article is fake, False otherwise.
        """
        try:
            # assert supports_response_schema(model=self.DEFAULT_LLM_MODEL)
            news_article = self.news_fetcher.get_news_article(url)
            if not news_article:
                print("Failed to fetch the news article.")
                raise Exception("Failed to fetch the news article.")

            user_prompt = f"""
                Detect misinformation in the news article.
                Article: '''{news_article}'''.
                Make sure the following are included:
                - Misinformation score: The likelihood of misinformation.
                - Explanation: Explanation of the score.
                - Linguistic features: The linguistic features that indicate misinformation.
                - Fact check: The fact check results.
            """
            response = completion(
                model=self.model,
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt,
                    }
                ],
                response_format=MisinformationDetectionResponse,
            )
            result_json = response.choices[0].message.content
            result = json.loads(result_json)
            return MisinformationDetectionResponse(**result)
        except Exception as e:
            print(f"Failed to check if the news article is fake: {e}")
            traceback.print_exc()
            raise e

## Step 7: Misinformation Detection Implementation

- **NewsFetcher**: A mocked class that simulates fetching article content from a URL.

- **MisinformationDetector**: Uses an LLM model to detect misinformation in the fetched article, providing a score, explanation, linguistic features, and fact-check results.

- **Execution**: The script creates an instance of `MisinformationDetector`, calls the `detect_misinformation` method with a URL, and prints the analysis result.

The `NewsFetcher` class was added to simulate article fetching.


In [ ]:
detector = MisinformationDetector()

url = "https://indianexpress.com/article/education/cbse-dual-board-exam-class-10-2026-9856208/"

response = detector.detect_misinformation(url)

print(response)

## Conclusion:

- The Misinformation Detection and Fact-Checking System leverages advanced language models to detect misinformation in news articles. The system includes several steps:

    1. `Environment Setup`: Installs necessary dependencies and validates environment variables.
    
    2. `Article Fetching`: Uses web scraping to retrieve news articles and verifies their authenticity.
    
    3. `Fake News Detection`: Analyzes articles using an LLM to determine if the content is fake.
    
    4. `Misinformation Detection`: Provides a comprehensive analysis, including a misinformation score, explanation, linguistic features, and fact-checking results.

- This system helps identify and verify news content, aiding in the fight against misinformation.


---

# Thank You for visiting The Hackers Playbook! 🌐

If you liked this research material;

- [Subscribe to our newsletter.](https://thehackersplaybook.substack.com)

- [Follow us on LinkedIn.](https://www.linkedin.com/company/the-hackers-playbook/)

- [Leave a star on our GitHub.](https://www.github.com/thehackersplaybook)

<div style="display:flex; align-items:center; padding: 50px;">
<p style="margin-right:10px;">
    <img height="200px" style="width:auto;" width="200px" src="https://avatars.githubusercontent.com/u/192148546?s=400&u=95d76fbb02e6c09671d87c9155f17ca1e4ef8f21&v=4"> 
</p>
</div>